# House Prices - Advanced Regression Techniques

https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques

Kaggle competition for predicting house prices. What follows is my attempt a implementing a data science project from start to finish.

In [6]:
import kagglehub
# from .autonotebook import tqdm as notebook_tqdm

C:\Users\zak\Projects\PyCharmProjects\ds-tutor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import os
import pandas as pd

In [8]:
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

# 2. Load the downloaded CSVs into pandas DataFrames
df_train = pd.read_csv(os.path.join(path, 'train.csv'))
df_test = pd.read_csv(os.path.join(path, 'test.csv'))

In [9]:
X = df_train.drop('SalePrice', axis=1)
y = df_train['SalePrice']

## Plan
ALWAYS USE SKLEARN PIPELINES.

My general approach is to create the most naive model I can and then iterate from there.

Things to consider
I'll test for the remove any missing values. train a model, then evaluate its perfromance.

In [10]:
# First define any transformers I will need that aren't part of the the base sklearn package.

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

class DropMissingColumns(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Find columns that do NOT have any NaNs during training
        # np.isnan(X).any(axis=0) checks each column for NaNs
        # The ~ symbol inverts it, keeping only the "clean" columns
        self.valid_cols_ = ~np.isnan(X).any(axis=0)
        return self

    def transform(self, X):
        # Apply that exact same column mask to any new data
        return X[:, self.valid_cols_]

In [11]:
# Next, define the simplest pipeline with the simplest model

from sklearn.compose import ColumnTransformer, make_column_selector, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

select_numeric = ColumnTransformer([
    ('num_filter', 'passthrough', make_column_selector(dtype_include=np.number))
])

naive_pipeline = Pipeline(
    steps = [
    ('drop_non_numeric_columns', select_numeric),
    ('drop_missing_columns', DropMissingColumns()),
    ('impute_test_missing', SimpleImputer(strategy='median')),
    ('linear_regression', LinearRegression())
    ],
    verbose=True)

model = TransformedTargetRegressor(
    regressor=naive_pipeline,
    transformer = None,
    func=np.log1p,
    inverse_func=np.expm1
)

In [12]:
# Cross-validate the model
from sklearn.model_selection import cross_validate

cross_validate(model, X, y, return_train_score=True)

[Pipeline]  (step 1 of 4) Processing drop_non_numeric_columns, total=   0.0s
[Pipeline]  (step 2 of 4) Processing drop_missing_columns, total=   0.0s
[Pipeline]  (step 3 of 4) Processing impute_test_missing, total=   0.0s
[Pipeline] . (step 4 of 4) Processing linear_regression, total=   0.0s
[Pipeline]  (step 1 of 4) Processing drop_non_numeric_columns, total=   0.0s
[Pipeline]  (step 2 of 4) Processing drop_missing_columns, total=   0.0s
[Pipeline]  (step 3 of 4) Processing impute_test_missing, total=   0.0s
[Pipeline] . (step 4 of 4) Processing linear_regression, total=   0.0s
[Pipeline]  (step 1 of 4) Processing drop_non_numeric_columns, total=   0.0s
[Pipeline]  (step 2 of 4) Processing drop_missing_columns, total=   0.0s
[Pipeline]  (step 3 of 4) Processing impute_test_missing, total=   0.0s
[Pipeline] . (step 4 of 4) Processing linear_regression, total=   0.0s
[Pipeline]  (step 1 of 4) Processing drop_non_numeric_columns, total=   0.0s
[Pipeline]  (step 2 of 4) Processing drop_mi

{'fit_time': array([0.01730704, 0.0090642 , 0.00882125, 0.00876617, 0.00855255]),
 'score_time': array([0.0019362 , 0.00177932, 0.00170159, 0.00165987, 0.00168777]),
 'test_score': array([ 0.89044458,  0.71835575,  0.89072022,  0.87639192, -0.99664195]),
 'train_score': array([0.76615238, 0.77569886, 0.76212424, 0.77344081, 0.84737023])}

### Submitting to Kaggle

In [13]:
nm = model.fit(X, y)
y_pred = nm.predict(df_test)

[Pipeline]  (step 1 of 4) Processing drop_non_numeric_columns, total=   0.0s
[Pipeline]  (step 2 of 4) Processing drop_missing_columns, total=   0.0s
[Pipeline]  (step 3 of 4) Processing impute_test_missing, total=   0.0s
[Pipeline] . (step 4 of 4) Processing linear_regression, total=   0.0s


In [14]:
submission_df = pd.DataFrame({'Id': df_test['Id'], 'SalePrice': y_pred})
submission_df.to_csv('submission.csv', index=False, header=True)

In [15]:
from kaggle.api.kaggle_api_extended import KaggleApi

# 1. Authenticate
api = KaggleApi()
api.authenticate()

# 2. Define competition and submission details
COMPETITION = 'house-prices-advanced-regression-techniques'

def get_latest_score(competition):
    # Fetch the list of all your submissions
    submissions = api.competition_submissions(competition)

    if submissions:
        latest = submissions[0]
        # Status will be 'pending' while Kaggle is still calculating
        return latest.public_score, latest.date, latest.status
    return None

In [16]:
get_latest_score(COMPETITION)
!kaggle competitions submit -c house-prices-advanced-regression-techniques -f submission.csv -m "My first model submission"

Successfully submitted to House Prices - Advanced Regression Techniques



  0%|          | 0.00/35.1k [00:00<?, ?B/s]
100%|██████████| 35.1k/35.1k [00:00<00:00, 91.8kB/s]


In [17]:
!kaggle competitions submissions -c house-prices-advanced-regression-techniques

fileName          date                        description                             status                     publicScore  privateScore  
----------------  --------------------------  --------------------------------------  -------------------------  -----------  ------------  
submission.csv    2026-03-02 15:13:38.300000  My first model submission               SubmissionStatus.PENDING                              
submission_2.csv  2026-03-01 11:05:40.970000  Prediction with upgraded preprocessing  SubmissionStatus.COMPLETE  0.14772                    
submission.csv    2026-02-28 14:03:42.350000  My first model submission               SubmissionStatus.COMPLETE  0.14772                    
submission.csv    2026-02-26 15:17:09.780000  My first model submission               SubmissionStatus.COMPLETE  0.14772                    
submission.csv    2026-02-26 12:01:50.070000  My first model submission               SubmissionStatus.COMPLETE  0.14772                    
submission.cs

Estimators, Transformers and Predictors

Scikit-learn is famous for its clean, consistent API. Almost every object in the library falls into one (or more) of three categories: Estimators, Transformers, and Predictors.
Here is a breakdown of the core methods that power this workflow, especially when using pipelines.
1. fit(X, y): The Learning Phase
Every algorithm in scikit-learn implements a fit method. This is where the "learning" happens.
•
What it does: It looks at the training data (X and sometimes y) to calculate internal parameters or state.
•
Important Rule: fit should never change the data itself. It only calculates and stores values inside the object. By convention, scikit-learn stores these learned parameters with a trailing underscore (e.g., self.valid_cols_ in your custom transformer, or model.coef_ in Linear Regression).
•
When it's used: Only once, during training.
2. transform(X): The Data Processing Phase
Objects that modify data (like imputers, scalers, or your DropMissingColumns) are called Transformers. They implement transform.
•
What it does: It takes new data X and applies the rules or parameters learned during the fit phase to modify and return the data.
•
Important Rule: transform relies completely on the state saved by fit. It does not learn anything new. This ensures your test data is treated exactly the same way as your training data.
•
When it's used: Whenever you need to process data (both during training and when making future predictions).
3. fit_transform(X, y): The Convenience Method
Most transformers also have a fit_transform method (which you get for free by inheriting from TransformerMixin).
•
What it does: It does exactly what it sounds like—it calls fit() and then immediately calls transform() on the same data.
•
Why it exists: It is often more computationally efficient to calculate parameters and apply them at the same time, rather than doing two separate passes over the data.
4. predict(X): The Decision Phase
Objects that make predictions (like Linear Regression, Random Forests, etc.) are called Predictors. They implement predict.
•
What it does: It takes fully pre-processed data X and uses the model parameters learned during fit to generate predictions (y_pred).
•
Related methods: Many classifiers also have predict_proba(X) to output the probability of each class, rather than just the final prediction.
How it all comes together in a Pipeline
The true magic of scikit-learn happens when you put these steps into a Pipeline. A Pipeline acts as a single, master object that orchestrates calling the right methods at the right time.
Let's look at what happens behind the scenes in your pipeline:
When you call pipeline.fit(X_train, y_train):
The pipeline orchestrates a chain reaction, passing the data from one step to the next:
1.
Step 1 (drop_missing_columns): Calls fit_transform(X_train) -> Learns which columns to keep, drops the bad ones, and passes the modified data to Step 2.
2.
Step 2 (impute_test_missing): Calls fit_transform(X_train_modified) -> Learns the medians, applies them, and passes the further modified data to Step 3.
3.
Step 3 (linear_regression): Calls fit(X_train_fully_processed, y_train) -> The final step is a predictor, so it only gets fitted. It learns the regression coefficients and stops.
When you call pipeline.predict(X_test):
The pipeline switches strictly to "application" mode. It never calls fit here, preventing "data leakage" (accidentally learning from your test data).
1.
Step 1 (drop_missing_columns): Calls transform(X_test) -> Drops the exact columns it memorized during training. Passes the data to Step 2.
2.
Step 2 (impute_test_missing): Calls transform(X_test_modified) -> Fills any new NaNs using the exact medians it memorized during training. Passes the data to Step 3.
3.
Step 3 (linear_regression): Calls predict(X_test_fully_processed) -> Uses its learned coefficients to make the final price predictions.
This orchestrated dance guarantees that any raw data you feed into predict() will undergo the exact same gauntlet of transformations as your training data did, preventing shape mismatches and ensuring realistic performance evaluations!



## What next?
At this stage the most naive model has been trained on the simplest possible form of the data. Categorical and missing data has been "handled" by dropping it.

We have some test and train scores.

Does the model have high bias and/or variance?
Are there specific rows the model does poorly on?

In [18]:
import numpy as np

from sklearn.compose import ColumnTransformer, make_column_selector, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Define specific transformers for each data type
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 2. Use ColumnTransformer to apply them to the right columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, make_column_selector(dtype_include=np.number)),
        ('cat', categorical_transformer, make_column_selector(dtype_include=object))
    ])

pipeline_2 = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('linear_regression', LinearRegression())
    ],
    verbose=True)

model_2 = TransformedTargetRegressor(
    regressor=pipeline_2,
    transformer=None,
    func=np.log1p,
    inverse_func=np.expm1
)
# Cross-validate the model
from sklearn.model_selection import cross_validate

cross_validate(model_2, X, y, return_train_score=True)

[Pipeline] ........ (step 1 of 2) Processing preprocess, total=   0.0s
[Pipeline] . (step 2 of 2) Processing linear_regression, total=   0.1s
[Pipeline] ........ (step 1 of 2) Processing preprocess, total=   0.0s
[Pipeline] . (step 2 of 2) Processing linear_regression, total=   0.1s
[Pipeline] ........ (step 1 of 2) Processing preprocess, total=   0.0s
[Pipeline] . (step 2 of 2) Processing linear_regression, total=   0.1s
[Pipeline] ........ (step 1 of 2) Processing preprocess, total=   0.0s
[Pipeline] . (step 2 of 2) Processing linear_regression, total=   0.1s
[Pipeline] ........ (step 1 of 2) Processing preprocess, total=   0.0s
[Pipeline] . (step 2 of 2) Processing linear_regression, total=   0.1s


{'fit_time': array([0.10021806, 0.11047721, 0.10772371, 0.1153419 , 0.09195685]),
 'score_time': array([0.01015353, 0.01017547, 0.00946856, 0.0095098 , 0.00963569]),
 'test_score': array([ 0.89395582,  0.63836742,  0.88091106,  0.92902664, -1.49846549]),
 'train_score': array([0.95095575, 0.96169396, 0.96034914, 0.95040406, 0.94891338])}